# M.I.N.E.R.V.A. v2.1 — From-Scratch Colab Run

**Misinformation Investigation through Networked Embeddings for Rumor Verification and Awareness**
FEU Institute of Technology IT thesis 2026

This notebook runs the full pipeline end-to-end starting from a fresh Colab session.

**v2.1 fixes (applied since v2.0 first release):**
1. Script 13 reads `p_*_fake` from script 12's actual top-level schema (was looking in wrong place — caused all UNCERTAIN verdicts)
2. Truncation gate is lenient — only flags genuinely broken fragments (was rejecting 94% of valid GPT-2 output)
3. Schema validator allows GPT-2 outputs without terminal punctuation
4. Faithfulness audit lexicon expanded to match actual response-bank prose
5. Drive mount is genuinely optional with smart download fallback

**Pipeline:**
```
[Setup → tests] → [Train detectors → train GPT-2] → [Generate posts → score]
              → [v2.0 explainability layer (18→27)] → [Final deck]
```

## 1. Configuration

Edit this cell once. Everything else uses these values.

In [ ]:
# Repo
REPO_URL    = "https://github.com/robertgeraldsenasin/MINERVA.git"
BRANCH_NAME = "upgrade/minerva-election-theme"
WORKDIR     = "/content"
REPO_DIR    = f"{WORKDIR}/MINERVA"

# Drive (optional — set to False if your account times out)
USE_DRIVE        = True
DRIVE_INPUT_DIR  = "/content/drive/MyDrive/MINERVA_outputs_revised"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MINERVA_v2_outputs"

# Pipeline scale
N_GENERATE_FAKE  = 500    # GPT-2 samples per class (legacy default)
N_GENERATE_REAL  = 500
DAYS_IN_DECK     = 7
CARDS_PER_DAY    = 8
MIN_CREDIBLE_PER_DAY = 3

print(f"Repo:    {REPO_URL}")
print(f"Branch:  {BRANCH_NAME}")
print(f"Workdir: {REPO_DIR}")
print(f"Drive:   {'enabled' if USE_DRIVE else 'DISABLED'}")

## 2. Mount Drive (gracefully — continues if it fails)

If Drive mount times out, the notebook falls back to runtime-only mode. **Setting `USE_DRIVE = False` in cell 1 skips this entirely.**

In [ ]:
DRIVE_MOUNT_OK = False

if USE_DRIVE:
    from google.colab import drive
    import os
    print("Hosted Colab runtime:", os.path.exists("/var/colab/hostname"))
    print("Attempting to mount Drive (5-min timeout)...")
    try:
        drive.mount("/content/drive", force_remount=True, timeout_ms=300000)
        DRIVE_MOUNT_OK = True
        print("✓ Drive mounted")
    except Exception as e:
        print(f"✗ Drive mount failed: {e!r}")
        print("  Continuing in runtime-only mode.")
        print("  At the end you'll get a downloadable zip instead of Drive save.")
else:
    print("Drive disabled by configuration (USE_DRIVE=False).")
    print("All outputs will be downloaded as a zip at the end.")

print(f"\nDRIVE_MOUNT_OK = {DRIVE_MOUNT_OK}")

## 3. Clone the upgrade branch from GitHub

This always works — no Drive dependency.

In [ ]:
import os, shutil

if os.path.exists(REPO_DIR):
    print(f"Removing existing {REPO_DIR}...")
    shutil.rmtree(REPO_DIR)

!git clone --branch "$BRANCH_NAME" "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!git log -1 --oneline
print()
print("Top-level files:")
!ls

## 4. Verify v2.x files arrived

In [ ]:
from pathlib import Path

REQUIRED = [
    "scripts/minerva_indicators.py",
    "scripts/minerva_response_bank.py",
    "scripts/minerva_schemas.py",
    "scripts/minerva_candidates.py",
    "scripts/minerva_filters.py",
    "scripts/13_score_generated_with_qlattice.py",
    "scripts/18_verdict_explain.py",
    "scripts/22_pseudonymize_entities.py",
    "scripts/23_enforce_election_theme.py",
    "scripts/24_curate_teaching_cards.py",
    "scripts/25_build_candidate_scenarios.py",
    "scripts/26_faithfulness_audit.py",
    "scripts/27_response_bank_versioning.py",
    "templates/candidate_profiles_three_candidates.json",
    "templates/teaching_response_bank_v1.json",
    "tests/test_indicators.py",
    "tests/test_filters.py",
]
missing = [p for p in REQUIRED if not Path(p).exists()]
if missing:
    print(f"✗ MISSING {len(missing)} files:")
    for m in missing:
        print(f"  {m}")
    raise RuntimeError("Cannot proceed with missing files")
print(f"✓ All {len(REQUIRED)} v2.x files present")

## 5. Install dependencies

In [ ]:
# Clean PEFT/TRL — MINERVA detector training does not need them
!python -m pip uninstall -y peft trl

# Upgrade packaging tools
!python -m pip install --upgrade -q pip setuptools wheel

# Repo's pinned stack
!python -m pip install -q -r requirements.txt

# v2.x dependencies
!python -m pip install -q pydantic pytest

# Runtime extras used by patched MINERVA scripts
!python -m pip install -q networkx spacy feyn torch-geometric

# Version sanity
import importlib.metadata as md
for pkg in ["transformers", "torch", "pydantic", "feyn", "spacy"]:
    try:
        print(f"  {pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg}: NOT INSTALLED")

## 6. spaCy model + working folders

In [ ]:
!python -m spacy download en_core_web_sm

from pathlib import Path
for d in ["data", "models", "generated", "reports", "outputs"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Created/verified: data, models, generated, reports, outputs")

## 7. Sanity check — run all 38 unit tests

If any test fails, **stop and fix** before continuing. The downstream pipeline relies on these passing.

In [ ]:
!python -m pytest tests/ -v

## 8. Inspect v2.x building blocks

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))

from minerva_response_bank import bank_stats, BANK_VERSION
from minerva_candidates import REGISTRY
import json

print("=== Response Bank ===")
print(json.dumps(bank_stats(), indent=2))
print()
print("=== Candidate Registry ===")
for code, c in REGISTRY.items():
    print(f"\n{code} — {c.name} ({c.archetype})")
    print(f"  Region: {c.region}, Party: {c.party_acronym}")
    print(f"  Slogan: {c.platform_slogan}")

## 9. Skip-or-train decision

If a previous run saved trained models to Drive, copy them in and skip cells 10-13.

In [ ]:
import os, shutil
from pathlib import Path

SKIP_TRAINING = False
SKIP_GENERATION = False

if DRIVE_MOUNT_OK and os.path.exists(DRIVE_INPUT_DIR):
    print(f"Found Drive input dir: {DRIVE_INPUT_DIR}")

    src_models = Path(DRIVE_INPUT_DIR) / "models"
    if src_models.exists():
        print(f"Copying models from {src_models}...")
        shutil.copytree(src_models, "models", dirs_exist_ok=True)
        SKIP_TRAINING = True
        print("✓ Models copied → SKIP_TRAINING=True")

    src_gen = Path(DRIVE_INPUT_DIR) / "generated"
    if src_gen.exists():
        shutil.copytree(src_gen, "generated", dirs_exist_ok=True)
        if Path("generated/gpt2_synthetic_samples_fake.jsonl").exists() and \
           Path("generated/gpt2_synthetic_samples_real.jsonl").exists():
            SKIP_GENERATION = True
            print("✓ Generated samples found → SKIP_GENERATION=True")

    src_data = Path(DRIVE_INPUT_DIR) / "data"
    if src_data.exists():
        shutil.copytree(src_data, "data", dirs_exist_ok=True)
        print(f"✓ Data copied")
else:
    if not DRIVE_MOUNT_OK:
        print("Drive not mounted → cells 10-13 will run training from scratch (~3 hours).")
    else:
        print(f"Drive mounted but {DRIVE_INPUT_DIR} not found → training from scratch.")

print(f"\nSKIP_TRAINING   = {SKIP_TRAINING}")
print(f"SKIP_GENERATION = {SKIP_GENERATION}")

## 10. Train detectors

⚠️ Multi-hour GPU training. Skipped if models came from Drive.

In [ ]:
if SKIP_TRAINING:
    print("⏭️  Skipping detector training (models loaded from Drive)")
else:
    !python scripts/01_download_dataset.py
    !python scripts/02_prepare_dataset.py
    !python scripts/03_split_dataset.py
    !python scripts/17_run_5seeds_detectors.py --run_id RUN1
    !python scripts/export_best_detectors_by_val.py --run_id RUN1
    !python scripts/06_extract_features.py
    !python scripts/07_train_random_forest.py
    !python scripts/08_train_qlattice.py
    !python scripts/09_train_degnn.py
    print("\n✓ Detectors trained")

## 11. Train GPT-2 generator

In [ ]:
if SKIP_TRAINING:
    print("⏭️  Skipping GPT-2 fine-tune")
else:
    !python scripts/10_prepare_gpt2MINERVA.py
    !python scripts/11_train_gpt2MINERVA.py
    print("\n✓ GPT-2 fine-tuned")

## 12. Patch + run GPT-2 generation

The legacy tokenizer fix is applied here, then we generate fake + real synthetic posts.

In [ ]:
if SKIP_GENERATION:
    print("⏭️  Skipping GPT-2 generation (samples loaded from Drive)")
else:
    # Apply tokenizer patch
    from pathlib import Path
    p = Path("scripts/12_generate_gpt2MINERVA.py")
    text = p.read_text(encoding="utf-8")
    old1 = '    gen_tok = AutoTokenizer.from_pretrained(str(paths.gpt2_dir), use_fast=True)\n'
    new1 = (
        '    gen_tok = AutoTokenizer.from_pretrained(str(paths.gpt2_dir), use_fast=True)\n'
        '    gen_tok.padding_side = "left"\n'
        '    if gen_tok.pad_token is None:\n'
        '        gen_tok.pad_token = gen_tok.eos_token\n'
    )
    if old1 in text and new1 not in text:
        text = text.replace(old1, new1, 1)
    old2 = '    gen_tok.add_special_tokens(special)\n'
    new2 = (
        '    gen_tok.add_special_tokens(special)\n'
        '    if gen_tok.pad_token is None:\n'
        '        gen_tok.pad_token = gen_tok.eos_token\n'
        '    gen_mdl.config.pad_token_id = gen_tok.pad_token_id\n'
    )
    if old2 in text and new2 not in text:
        text = text.replace(old2, new2, 1)
    text = text.replace('        pad_token_id=gen_tok.eos_token_id,\n',
                        '        pad_token_id=gen_tok.pad_token_id,\n', 1)
    p.write_text(text, encoding="utf-8")
    print(f"✓ Patched {p}")

    # Generate
    !python scripts/12_generate_gpt2MINERVA.py {N_GENERATE_FAKE} fake 0.70 120 --accept_mode ensemble3 --out generated/gpt2_synthetic_samples_fake.jsonl
    !python scripts/12_generate_gpt2MINERVA.py {N_GENERATE_REAL} real 0.70 120 --accept_mode ensemble3 --out generated/gpt2_synthetic_samples_real.jsonl
    print("\n✓ Generated synthetic posts")

## 13. Score generated posts with QLattice (v2.1 — bug-fixed)

**v2.1 fixes:** reads top-level `p_*_fake` fields correctly, lenient truncation gate.

In [ ]:
!python scripts/13_score_generated_with_qlattice.py \
    --in_file  generated/gpt2_synthetic_samples_fake.jsonl \
    --out_file generated/gpt2_synthetic_final_fake.jsonl \
    --rejection_log reports/score_rejection_log_fake.jsonl

!python scripts/13_score_generated_with_qlattice.py \
    --in_file  generated/gpt2_synthetic_samples_real.jsonl \
    --out_file generated/gpt2_synthetic_final_real.jsonl \
    --rejection_log reports/score_rejection_log_real.jsonl

# Concatenate
!cat generated/gpt2_synthetic_final_fake.jsonl generated/gpt2_synthetic_final_real.jsonl > generated/gpt2_synthetic_final_both.jsonl

import os
size = os.path.getsize('generated/gpt2_synthetic_final_both.jsonl')
print(f"\nFinal both size: {size:,} bytes")
assert size > 0, "ERROR: scoring produced empty output"
!wc -l generated/gpt2_synthetic_final_both.jsonl

## 14. Script 18 — Verdict + explanation

After this cell, watch the diversity log line — should be near 100%.

In [ ]:
!python scripts/18_verdict_explain.py \
    --in_file   generated/gpt2_synthetic_final_both.jsonl \
    --out_file  generated/unity_cards.json \
    --audit_out reports/audit_18.jsonl \
    --seed 1729

## 15. Script 21 — Balance unity cards

In [ ]:
!python scripts/21_balance_unity_cards.py \
    --in_file  generated/unity_cards.json \
    --out_file generated/unity_cards_balanced.json \
    --target_total 200 \
    --report_out reports/balance_report.json

## 16. Script 22 — Pseudonymise to 3 candidates

In [ ]:
!python scripts/22_pseudonymize_entities.py \
    --in_file  generated/unity_cards_balanced.json \
    --out_file generated/unity_cards_pseudonymized.json \
    --re_explain --seed 1729

## 17. Script 23 — Enforce electoral theme

In [ ]:
!python scripts/23_enforce_election_theme.py \
    --in_file  generated/unity_cards_pseudonymized.json \
    --out_file generated/unity_cards_themed.json \
    --theme_threshold 0.55 \
    --report_out reports/theme_filter_report.json \
    --rejection_log reports/theme_rejection_log.jsonl

## 18. Script 24 — Curate teaching cards

In [ ]:
!python scripts/24_curate_teaching_cards.py \
    --in_file  generated/unity_cards_themed.json \
    --out_file generated/story_cards.json \
    --reject_out generated/story_cards_rejected.json \
    --report_out reports/story_cards_curation_report.json \
    --days {DAYS_IN_DECK} --cards_per_day {CARDS_PER_DAY} --min_credible_per_day {MIN_CREDIBLE_PER_DAY} \
    --seed 1729

## 19. Script 25 — VERIdex candidate scenarios

In [ ]:
!python scripts/25_build_candidate_scenarios.py \
    --story_cards generated/story_cards.json \
    --out_file    generated/candidate_scenarios.json

## 20. Script 26 — Faithfulness audit (must be 100%)

In [ ]:
!python scripts/26_faithfulness_audit.py \
    --in_file       generated/story_cards.json \
    --report_out    reports/faithfulness_audit_report.json \
    --failures_out  reports/faithfulness_failures.jsonl

## 21. Script 27 — Stamp bank version

In [ ]:
!python scripts/27_response_bank_versioning.py stamp \
    --in_file  generated/story_cards.json \
    --out_file generated/story_cards_stamped.json

!python scripts/27_response_bank_versioning.py export \
    --out_file templates/teaching_response_bank_v1_export.json

## 22. Final reports dashboard

In [ ]:
import json, glob

for fp in sorted(glob.glob('reports/*.json')):
    print(f'\n{"="*60}')
    print(fp)
    print('='*60)
    print(json.dumps(json.load(open(fp)), indent=2)[:2000])

## 23. Sample cards from final deck

In [ ]:
import json

deck = json.load(open('generated/story_cards.json'))
if not deck:
    print("ERROR: empty deck. Check curation report and theme rejection log.")
else:
    print(f'Deck size: {len(deck)} cards across {max(c["day"] for c in deck)} days\n')
    seen = set()
    for c in deck:
        key = (c['verdict'], c['candidate'])
        if key in seen:
            continue
        seen.add(key)
        print(f"--- Day {c['day']} | {c['verdict']} | {c['candidate']} | tier={c['explanation']['tier']} ---")
        print(f"Text: {c['text'][:150]}...")
        print(f"Fired: {c['fired_indicators']}")
        print(f"Summary: {c['explanation']['summary'][:300]}")
        print()

## 24. Save outputs

- **Drive available:** save to `MINERVA_v2_outputs/` for next run's skip-training shortcut
- **No Drive:** zip downloads to your laptop. Upload to Drive manually after.

In [ ]:
if DRIVE_MOUNT_OK:
    import shutil
    from pathlib import Path

    drive_out = Path(DRIVE_OUTPUT_DIR)
    if drive_out.exists():
        shutil.rmtree(drive_out)

    KEEP = [
        "generated", "reports",
        "models/roberta_finetuned",
        "models/distilbert_multilingual_finetuned",
        "models/gpt2_tagalog_finetuned",
        "models/pca_roberta.joblib",
        "models/pca_distilbert.joblib",
        "models/degnn.pt",
        "models/degnn_artifacts.joblib",
        "models/qlattice_equation.txt",
        "models/best_qlattice.json",
        "models/qlattice",
        "templates",
    ]
    IGNORE = shutil.ignore_patterns(
        "checkpoint-*", "optimizer.pt", "scheduler.pt",
        "trainer_state.json", "rng_state.pth", "*.tmp",
    )
    for rel in KEEP:
        src = Path(rel)
        if not src.exists():
            print(f"[skip] {rel}")
            continue
        dst = drive_out / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=IGNORE)
        else:
            shutil.copy2(src, dst)
        print(f"[ok] {rel} → {dst}")
    print(f"\nDrive export complete: {drive_out}")
    print(f"\nFor Unity, copy these to your Android project:")
    print(f"  {drive_out}/generated/story_cards.json    → Chattr feed")
    print(f"  {drive_out}/generated/candidate_scenarios.json → VERIdex profiles")
else:
    # No Drive — zip and download deliverables
    import shutil, os
    from google.colab import files
    out_dir = '/tmp/minerva_v2_deliverables'
    os.makedirs(out_dir, exist_ok=True)
    for f in ['generated/story_cards.json',
              'generated/story_cards_stamped.json',
              'generated/candidate_scenarios.json',
              'generated/unity_cards_themed.json']:
        if os.path.exists(f):
            shutil.copy(f, f'{out_dir}/{os.path.basename(f)}')
    if os.path.exists('reports'):
        shutil.copytree('reports', f'{out_dir}/reports', dirs_exist_ok=True)
    if os.path.exists('templates'):
        shutil.copytree('templates', f'{out_dir}/templates', dirs_exist_ok=True)
    shutil.make_archive('/tmp/minerva_v2_deliverables', 'zip', out_dir)
    print('Downloading deliverables zip to your laptop...')
    files.download('/tmp/minerva_v2_deliverables.zip')
    print()
    print("After download:")
    print("  1. Drag the zip into your Drive manually if needed")
    print("  2. For Unity: extract story_cards.json + candidate_scenarios.json")

---

## Appendix A — Run-time estimates

| Section | Time | GPU? |
|---|---|---|
| 1–8 (setup) | ~3 min | No |
| 9 (skip decision) | <1 min | No |
| 10–11 (training) | 1.5–3 hrs | **Yes** |
| 12–13 (gen + score) | 20–40 min | **Yes** for 12, No for 13 |
| 14–21 (v2.x pipeline) | <2 min | No |
| 22–24 (reports + save) | <1 min | No |

**Cold start: ~3-4 hours.** With cached Drive models: ~10 minutes.

## Appendix B — Where to look if a stage fails

| Cell | First diagnostic |
|---|---|
| 2 (Drive) | If timeout: set `USE_DRIVE = False` in cell 1, re-run from cell 2 |
| 7 (tests) | If any fail, the error names which module — fix before continuing |
| 10 (training) | `!nvidia-smi` to confirm GPU allocated |
| 12 (generation) | If tokenizer error, re-run cell 12 (re-applies patch) |
| 13 (scoring) | `reports/score_rejection_log_*.jsonl` for reasons |
| 14 (explain) | Diversity should be near 100% — if low, check `reports/audit_18.jsonl` |
| 17 (theme) | `reports/theme_rejection_log.jsonl` for per-card reasons |
| 18 (curate) | `generated/story_cards_rejected.json` |
| 20 (audit) | `reports/faithfulness_failures.jsonl` if pass <100% |